# 노트북 05 — 우리가 생략한 것, 그리고 그 이유

노트북 03은 날카로운 결론으로 끝났습니다: 체인 키가 한 번 노출되면 이후의 모든 메시지를 읽을 수 있다. 이는 이 교육용 토이와 실제 배포 가능한 종단 간 메신저 사이에 존재하는 *여러* 격차 중 *하나*에 불과합니다. 이 챕터는 그 격차들을 정리하고, 각각이 실제로 어떤 위협에 대응하는지 설명하며, 메우는 데 어느 정도의 비용이 드는지 스케치합니다.

## 1. DH 래칫 (post-compromise security)

**현재 우리가 가진 것**: 대칭 래칫뿐. `chain_key`가 한 번 노출되면 이후 메시지가 영구적으로 노출됩니다 — 노트북 03 5단계 참조.

**실제 Signal이 가진 것**: *Double* Ratchet. 대화의 방향이 바뀔 때마다(Alice→Bob, 그리고 Bob→Alice), 송신할 측이 새로운 X25519 키쌍을 생성하고 공개키를 메시지에 첨부한 뒤, 새로 계산된 DH 공유 비밀을 양쪽 체인 키에 섞어 넣습니다. 공격자가 본 적 없는 키에서 새로운 엔트로피가 시스템에 들어오므로 — 체인이 *치유(heal)* 됩니다.

**추가 비용**: 약 150줄.
1. `Session`에 `dh_send_priv`와 `dh_recv_pub`를 추가.
2. 매번 송신 시, 새로운 "송신 burst"라면 새 X25519 키쌍을 굴리고, 새 루트 키 유도(`HKDF(root_key, dh_output)`), 체인 키 재설정, 새 공개키 첨부.
3. 수신 시, 인식되지 않은 DH 공개키가 도착하면 대칭 단계를 거꾸로(미러) 실행.

**우리가 생략한 이유**: 상태 머신 복잡도가 두 배가 되고, 대칭 래칫의 역할이 가려집니다. 노트북 03은 대칭 전용의 한계를 *눈에 보이게* 만듭니다; 실제 제품에서는 DH 절반 없이는 절대 출시하지 않을 것입니다.

## 2. 인증 (signed prekey, identity 바인딩)

**현재 우리가 가진 것**: TOFU — Trust On First Use. Alice가 `bob.pub`를 import할 때, 그 바이트가 정말 Bob의 것이라고 신뢰합니다. 서명도, 지문(fingerprint) 비교도, 공개된 키 디렉터리도 없습니다.

**실제 Signal이 가진 것**:

1. **장기 신원 키(identity key)** — Ed25519, 절대 회전(rotate)되지 않음.
2. **서명된 prekey** — 중기(medium-term) X25519 키, identity 키로 서명됨. 매주 회전.
3. **일회성 prekey(one-time prekey)** — 서버에 업로드되는 1회용 X25519 키 묶음. 첫 메시지가 그 중 하나를 소비함으로써, 초기 핸드셰이크에 수신자 측에도 추가 전방 비밀성 계층이 생깁니다.
4. **Safety number** — 양쪽 identity 키의 지문을 사용자가 외부 채널로 비교하여 서버에 의한 MITM을 탐지할 수 있게 함.

**추가 비용**:
- 서명 방식 (Ed25519, 또는 완전 PQ 인증을 원한다면 ML-DSA / SLH-DSA — [자매 도서 ch. 10](https://hulryung.github.io/ml-kem-notebooks/ko/notebooks/10_signatures_and_friends.html) 참조).
- prekey 번들 포맷과 디렉터리 서비스 (또는 파일 큐 위에서 publish-and-fetch 동작).
- safety number 렌더링 함수.

**이것들이 없으면**, 컨택트 파일을 주입할 수 있는 공격자(사용자 노트북의 멀웨어, 파일 큐 변조 등)는 import 시점에 자기 키를 끼워 넣고 이후 모든 메시지를 MITM할 수 있습니다. TOFU는 *오직* 첫 키를 전달한 채널만큼만 강합니다.

## 3. 재전송 및 순서 변경 방어

**현재 우리가 가진 것**: `msg_index` 필드와, 수신 체인을 한 단계씩 전진시키는 세션. 같은 메시지의 사본 두 개가 도착하면, 두 번째 복호화는 잘못된 키(체인이 이미 전진)를 만들어 AEAD가 거부합니다. 그러나 순서가 어긋난 전송은 *복호화 자체를 깨뜨리고*, 수신자는 복구할 수 없습니다.

**실제 Signal이 가진 것**: 메시지마다 `(dh_pub, prev_chain_length, msg_index)` 헤더 포함. 수신자는 건너뛴 메시지 키들을 작은 캐시에 저장하여, 일정 윈도우 내의 순서 어긋남은 정상적으로 복호화합니다. 캐시 크기는 DoS 방지를 위해 제한됩니다.

**추가 비용**: 약 50줄 — 미리 유도하고, 건너뛴 키들을 `Session`에 저장하며, 크기 또는 시간 기준으로 제거.

**왜 중요한가**: 어떤 실제 네트워크든 순서가 바뀝니다. 건너뛴 키 처리가 없으면 UDP 패킷 한 개만 떨어져도 채널 전체가 막힙니다.

## 4. 메타데이터 프라이버시

**현재 우리가 가진 것**: `sender`, `recipient`, `sent_at`, 그리고 (첫 메시지의 경우) prekey 번들 크기까지 모두 디스크에 평문으로 저장됩니다. inbox 디렉터리를 읽을 수 있는 사람은 누구나 사회적 그래프와 타이밍을 볼 수 있습니다.

**실제 시스템들이 하는 것** (부분 목록 — 어느 것도 완전한 해법은 아닙니다):
- **Signal sealed sender**: 수신자의 서버는 메시지가 Bob에게 도착했다는 것은 알지만, 누가 보냈는지는 모릅니다. 송신자 신원이 Bob의 identity 키 아래에서 암호화됩니다.
- **Tor / 어니언 라우팅**: 네트워크 레벨 메타데이터를 숨깁니다. 대가는 지연.
- **패딩 & 커버 트래픽**: 메시지 길이와 타이밍을 숨깁니다 — 대역폭 비용이 막대해서 대규모로 적용한 곳은 없습니다.

**이 토이에 추가하는 비용**: sealed sender는 약 30줄(송신자 필드를 수신자 장기 키 아래에서 암호화). 네트워크 레벨 메타데이터 방어는 *어떤* 응용 프로토콜의 범위도 벗어납니다.

**솔직한 평가**: 메타데이터 누출은 메시징에서 가장 어려운 미해결 문제입니다.

## 5. 멀티 디바이스, 그룹 채팅, 비동기 전송

셋 다 만만치 않은 설계 문제입니다:

- **멀티 디바이스**: Alice가 폰과 노트북 둘 다 가지고 있으면, 같은 대화에서 양쪽 모두 송수신할 수 있어야 합니다. Signal은 각 디바이스를 자체 Signal identity로 만들고, 사용자의 마스터 키로 서명한 *device-list*와 모든 디바이스 쌍 사이의 pairwise 세션으로 해결합니다. 그룹 채팅은 그 위에서 "메시지를 N번 보내기"가 됩니다.
- **그룹 채팅**: 작은 N에서는 pairwise 세션이 동작합니다. 큰 N에서는 *Sender Keys*(그룹별 대칭 체인) 또는 MLS(RFC 9420 — 수천 명 멤버를 위한 트리 기반 키 합의)가 필요합니다. WhatsApp과 Discord가 이전한 표준이 MLS입니다.
- **비동기 전송**: 우리의 파일 큐는 단순히 비동기적입니다 — 수신자가 폴링만 합니다. 실제 시스템은 서버측 큐잉, 푸시 알림, 그리고 Alice가 한 달 오프라인일 때 송신자가 일회성 키가 떨어지지 않도록 *prekey 보충(replenishment)* 이 필요합니다.

각각이 독립된 챕터 분량입니다. 노트북 02-04의 양자(two-party), 온라인 전용, 단일 디바이스 토이가 가능한 가장 단순한 출발점입니다.

## 6. 구현 강화(hardening)

[자매 도서 노트북 09 "프로덕션 대비 부족한 점"](https://hulryung.github.io/ml-kem-notebooks/ko/notebooks/09_wrap_up.html)에서 그대로 상속된, 이 책에도 똑같이 적용되는 항목들:

- **Constant-time**: 우리의 ML-KEM 구현(`pqc_edu`)은 Python int를 사용하며, 많은 연산이 timing-variable입니다. 실제 구현은 constant-time C / Rust로 작성됩니다.
- **메모리 zeroization**: 중간 키들이 Python의 GC로 새어 나갑니다. 실제 구현은 `mlock`을 호출하고 버퍼를 명시적으로 와이프합니다.
- **사이드 채널**: 전력, EM, 캐시. 어떤 순수 파이썬 프로젝트의 범위도 벗어납니다.
- **KAT 검증**: `pqc_edu`는 NIST Known Answer Test에 대해 검증되지 않았습니다. 프로덕션 코드는 반드시 검증되어야 합니다.
- **난수 품질**: 우리는 `os.urandom`을 신뢰합니다. 실제 시스템은 엔트로피 체인을 처음부터 끝까지 감사합니다.

## 요약 표

| 속성                              | 이 토이             | 실제 Signal급 시스템             |
| --------------------------------- | ------------------- | -------------------------------- |
| 기밀성 (도청)                     | ✅                  | ✅                               |
| AEAD (변조 탐지)                  | ✅                  | ✅                               |
| 전방 비밀성 (과거 메시지)         | ✅                  | ✅                               |
| Post-compromise security          | ❌                  | ✅ (DH 래칫)                     |
| 신원 인증                         | ❌ TOFU만           | ✅ Ed25519 + safety #            |
| 순서 어긋남 허용                  | ❌                  | ✅ 건너뛴 키 캐시                |
| Sealed sender                     | ❌                  | ✅                               |
| 멀티 디바이스                     | ❌                  | ✅                               |
| 그룹 채팅                         | ❌                  | ✅ Sender Keys / MLS             |
| 비동기 + 오프라인 전송            | 부분적 (파일 폴링)  | ✅ 서버 + 푸시                   |
| Constant-time 암호                | ❌                  | ✅                               |
| KAT 검증된 KEM                    | ❌                  | ✅                               |

체크된 항목 8개 vs 4개. 우리가 *체크한* 4개는 *먼저* 제대로 잡아야 하는 것들입니다 — 기밀성, AEAD, 전방 비밀성 없이는 나머지가 모두 장식에 불과합니다. 하지만 실제 제품으로 출시하려면 나머지 모든 행을 메워야 합니다.

## 다음에 읽을 거리

- **Signal Double Ratchet 명세** — [signal.org/docs/specifications/doubleratchet](https://signal.org/docs/specifications/doubleratchet/). 30페이지의 정전(canonical) 자료.
- **Signal X3DH 명세** — [signal.org/docs/specifications/x3dh](https://signal.org/docs/specifications/x3dh/). prekey 메커니즘의 자세한 설명.
- **iMessage PQ3** — Apple의 블로그 포스트와 기술 명세. 처음으로 배포된 하이브리드 PQ 메신저.
- **MLS (RFC 9420)** — IETF의 그룹 메시징 표준. WhatsApp, Discord, Webex의 기반.
- **노트북 06** — pq-messenger와 실제 시스템들의 나란히 비교.